# Context Processor
> Utilities for expanding, compacting, and canonicalising JSON-LD.

Provides a robust processor that uses **pyld** while delegating all network fetches to our registry-aware, cache-memoised loader.

In [ ]:
#| default_exp core.context

## imports  


In [ ]:
#| export
from __future__ import annotations

import re, json, hashlib
from http import HTTPStatus
from functools import lru_cache
from typing import Any, Dict, List, Tuple

from pyld import jsonld

from cogitarelink.core.debug import get_logger
from cogitarelink.core.cache import InMemoryCache
from cogitarelink.vocab.registry import registry
from cogitarelink.vocab.composer import composer

log = get_logger("context")
_cache = InMemoryCache(maxsize=512)

In [ ]:
#| export
_LINK_RE = re.compile(r'<([^>]+)>;\s*rel="([^"]+)";\s*type="([^"]+)"')

##  registry-aware document loader

## URI resolution
- Step ① Try to dereference the URI (GET → follow redirects → 10 s timeout).
	- If the response is application/ld+json → return as-is.
	- If it’s Turtle (or any RDF serialisation rdflib can parse) → derive a
minimal prefix context.
	- If it’s HTML but the Link: header advertises an
application/ld+json alternate (e.g. Schema.org) → follow that link.
- Step ② If dereference fails (network error or unsupported format)
fallback to the registry lookup (registry.resolve(url)).
- Step ③ Still nothing? raise JsonLdError with a LLM-friendly
message and the chain of attempts in error.details.

Everything is fully cached with the same namespace-scoped memoizer you already
use, so repeated calls never hit the network twice.

In [ ]:
#| export
def _parse_link_header(hdr: str) -> List[Tuple[str, str, str]]:
    "Return list of (url, rel, type) triples from RFC 8288 Link header."
    return _LINK_RE.findall(hdr)

@_cache.memoize("jsonld-doc")
def _http_get(url: str) -> "httpx.Response":
    "GET with redirect, 10 s timeout, memoised by URL."
    import httpx
    log.debug(f"GET {url}")
    r = httpx.get(url, follow_redirects=True, timeout=10)
    r.raise_for_status()
    return r  # caller decides how to interpret

## dereference helper

In [ ]:
#| export
def _wrap_json_doc(text: str, url: str) -> Dict[str, Any]:
    return dict(contextUrl=None, documentUrl=url, document=text)

def _try_dereference(url: str) -> Dict[str, Any] | None:
    """
    Return a pyld‐style remote-doc dict if dereference succeeds,
    else None (caller will fall back to registry).
    """
    try:
        r = _http_get(url)
    except Exception as e:
        log.debug(f"network fetch failed for {url}: {e}")
        return None

    ctype = r.headers.get("content-type", "").split(";")[0].strip()
    body  = r.text

    # 1️⃣ JSON-LD or already a context
    if ctype in ("application/ld+json", "application/json"):
        try:
            json.loads(body)  # sanity check
            return _wrap_json_doc(body, url)
        except json.JSONDecodeError:
            pass  # keep inspecting

    # 2️⃣ Turtle → derive minimal context
    if ctype in ("text/turtle", "application/x-turtle"):
        from rdflib import Graph
        g = Graph().parse(data=body, format="turtle")
        ctx = {"@context": {p: str(iri) for p, iri in g.namespaces()}}
        return _wrap_json_doc(json.dumps(ctx), url)

    # 3️⃣ HTML with Link: header (Schema.org pattern)
    link_hdr = r.headers.get("Link")
    if link_hdr:
        for link, rel, typ in _parse_link_header(link_hdr):
            if rel == "alternate" and typ == "application/ld+json":
                return _try_dereference(link)  # recursive but cached
    return None

In [ ]:
#| export
def get_document_loader(network_first: bool = True):
    """
    Return a pyld‐compatible document loader.

    * `network_first=True`  – follow‐your-nose, registry is fallback.
    * `network_first=False` – current behaviour (registry first, then HTTP).
    """
    mem: Dict[str, Dict[str, Any]] = {}

    def loader(url: str):
        attempts = []

        def _push(res, label):
            if res:
                mem[url] = res
                return res
            attempts.append(label)
            return None

        # ① network
        if network_first:
            doc = _push(_try_dereference(url), "network")
            if doc: return doc

        # ② registry
        try:
            entry  = registry.resolve(url)
            ctx    = entry.context_payload()
            return _wrap_json_doc(json.dumps(ctx), url)
        except KeyError:
            attempts.append("registry")

        # ③ network second-chance (when registry_first mode)
        if not network_first:
            doc = _push(_try_dereference(url), "network")
            if doc: return doc

        # fail – raise LLM-friendly JsonLdError
        detail = {"attempts": attempts, "url": url}
        raise JsonLdError(
            f"Could not load linked-data document at <{url}>. "
            "Tried network fetch and registry fallback.",
            "jsonld.LoadDocumentError",
            code="loading document failed",
            details=detail
        )

    return loader

# register globally (network_first=True by default)
jsonld.set_document_loader(get_document_loader(network_first=True))

In [ ]:
#| hide
# validate Schema.org root URI resolves via Link: header
schema_ctx = get_document_loader()( "https://schema.org/" )
assert "@context" in json.loads(schema_ctx["document"])

## ContextProcessor  

In [ ]:
#| export
class ContextProcessor:
    "Thin wrapper around pyld with sensible defaults."

    def expand(self, doc: Dict[str, Any] | List[Any]) -> List[Dict[str, Any]]:
        "Return expanded JSON-LD array."
        return jsonld.expand(doc, options=dict(base=None))

    def compact(self, doc: Dict[str, Any] | List[Any],
                ctx: Dict[str, Any] | List[Any]) -> Dict[str, Any]:
        "Compact against a given `@context`."
        return jsonld.compact(doc, ctx, options=dict(base=None))

    def normalize(self, doc: Dict[str, Any] | List[Any]) -> str:
        """
        Canonicalise using URDNA2015 and return **N-Quads** string.
        Useful for later signing.
        """
        return jsonld.normalize(
            doc,
            options=dict(algorithm="URDNA2015", format="application/n-quads")
        )

    # convenience ------------------------------------------------------
    def expand_with(self, prefixes: List[str], doc: Dict[str, Any]) -> List[Dict[str, Any]]:
        "Inject composed context, then expand."
        ctx = composer.compose(prefixes)
        merged = dict(doc, **ctx)
        return self.expand(merged)
        
    def propagates(self, context):
        """Determine if a context propagates to child nodes (JSON-LD 1.1 feature)"""
        if isinstance(context, dict) and "@propagate" in context:
            return context["@propagate"] is not False
        return True  # Default is to propagate

    def apply_scoped_contexts(self, entity, types=None):
        """Apply type-scoped contexts based on entity @type"""
        if not isinstance(entity, dict):
            return entity
        
        # Get entity types if not provided
        if types is None:
            if "@type" not in entity:
                return entity
            types = entity["@type"] if isinstance(entity["@type"], list) else [entity["@type"]]
        
        if "@context" not in entity:
            return entity
        
        ctx = entity["@context"]
        
        # Check if context has type-scoped definitions
        if not isinstance(ctx, dict) or "@type" not in ctx:
            return entity
        
        type_scopes = ctx["@type"]
        if not isinstance(type_scopes, dict):
            return entity
        
        # Find matching scoped contexts for entity types
        matching_contexts = []
        for t in types:
            if t in type_scopes:
                scoped_ctx = type_scopes[t]
                if scoped_ctx:
                    matching_contexts.append(scoped_ctx)
        
        if not matching_contexts:
            return entity
        
        # Create a new entity with merged context
        result = entity.copy()
        base_ctx = ctx.copy()
        del base_ctx["@type"]  # Remove type scopes to avoid recursion
        
        # Apply found contexts
        if len(matching_contexts) == 1:
            result["@context"] = [base_ctx, matching_contexts[0]]
        else:
            result["@context"] = [base_ctx] + matching_contexts
        
        return result

## quick tests  

In [ ]:
#| hide
from fastcore.test import test_eq
import pytest

proc = ContextProcessor()

# 4.1 basic expand
example = {"@context": {"name": "http://schema.org/name"}, "name": "Ada"}
exp = proc.expand(example)
assert isinstance(exp, list), "Expected expanded result to be a list"
test_eq(exp[0]["http://schema.org/name"][0]["@value"], "Ada")

# 4.2 cache verifies: second call should hit memo
url = "https://schema.org/docs/jsonldcontext.jsonld"
_first  = get_document_loader()(url)        # fetch & cache
_second = get_document_loader()(url)        # cached
test_eq(_first["document"], _second["document"])

In [ ]:
#| hide
# 1. Test compact method
compact_example = {
    "@id": "http://example.org/person",
    "http://schema.org/name": [{"@value": "Ada Lovelace"}]
}
context = {"@context": {"name": "http://schema.org/name"}}
compacted = proc.compact(compact_example, context)
test_eq(compacted["name"], "Ada Lovelace")

# 2. Test normalize method
norm_example = {"@context": {"name": "http://schema.org/name"}, "name": "Ada"}
normalized = proc.normalize(norm_example)
assert isinstance(normalized, str), "Expected normalized result to be a string"
assert "http://schema.org/name" in normalized, "Expected normalized result to contain expanded predicate"


In [ ]:
#| hide
# 3. Test expand_with method
# Assuming your registry has some predefined prefixes like 'schema'
simple_doc = {"name": "Ada Lovelace"}
expanded = proc.expand_with(["schema"], simple_doc)

# Check that expansion happened with the schema context
assert isinstance(expanded, list), "Expected expanded result to be a list"
assert "http://schema.org/name" in expanded[0], "Expected expanded document to use schema.org context"


## protected-term validator

In [ ]:
#| export
def validate_context(ctx: Dict[str, Any]) -> None:
    """
    Validate a JSON-LD 1.1 context for:
    - Protected term redefinition (§4.4.5)
    - Type-scoped context validity
    - @nest directives
    
    Raises ValueError if validation fails.
    """
    if "@context" in ctx:
        ctx = ctx["@context"]
    if not isinstance(ctx, list):
        ctx = [ctx]

    seen: Dict[str, Dict] = {}
    nested_props = set()
    
    for c in ctx:
        if not isinstance(c, dict):   # remote URL – skip (loader enforces)
            continue
            
        # Check for @nest values
        for term, value in c.items():
            if value == "@nest" or (isinstance(value, dict) and value.get("@id") == "@nest"):
                nested_props.add(term)
                
            # Check for type-scoped contexts
            if term == "@type" and isinstance(value, dict):
                for type_key, type_ctx in value.items():
                    # Recursively validate type-scoped contexts
                    if isinstance(type_ctx, dict):
                        validate_context({"@context": type_ctx})
        
        # Check protected term reuse
        for k, v in c.items():
            is_protected = (isinstance(v, dict) and v.get("@protected") is True) or (
                "@protected" in c and c["@protected"] is True and 
                not (isinstance(v, dict) and v.get("@protected") is False)
            )
            
            if is_protected and k in seen and seen[k] is not v:
                raise ValueError(f"Protected term '{k}' re-defined in context")

            if is_protected:
                seen[k] = v

In [ ]:
validate_context(composer.compose(["vc","epcis"]))

In [ ]:
#| export
def protected_terms(ctx):
    """Extract protected terms from a JSON-LD 1.1 context"""
    protected = set()
    
    if not isinstance(ctx, dict):
        return protected
    
    # Check for context-wide protection
    ctx_protected = ctx.get("@protected", False)
    
    for term, defn in ctx.items():
        if term.startswith("@"):
            continue
            
        # Check explicit protection
        if isinstance(defn, dict) and "@protected" in defn:
            if defn["@protected"]:
                protected.add(term)
        # Check implicit protection from context-wide setting
        elif ctx_protected:
            is_protected = True
            if isinstance(defn, dict) and "@protected" in defn:
                is_protected = defn["@protected"]
                
            if is_protected:
                protected.add(term)
    
    return protected

In [ ]:
#| export
def process_nest(ctx, data):
    """Process and apply @nest directives in a context to the data"""
    if not isinstance(data, dict):
        return data
    
    result = data.copy()
    
    # Find properties marked with @nest
    nested_props = []
    if isinstance(ctx, dict):
        for term, value in ctx.items():
            if value == "@nest" or (isinstance(value, dict) and value.get("@id") == "@nest"):
                nested_props.append(term)
    
    # Extract properties from nested objects
    for nest_term in nested_props:
        if nest_term in result and isinstance(result[nest_term], dict):
            # Move properties from nested object to parent
            for prop, value in result[nest_term].items():
                if prop not in result:  # Don't override existing properties
                    result[prop] = value
            # Remove the nest container
            del result[nest_term]
    
    return result

In [ ]:
#| hide
# Test new methods

# Test protected_terms
ctx_with_protected = {"@context": {"@protected": True, "name": "http://example.org/name"}}
assert "name" in protected_terms(ctx_with_protected["@context"])

ctx_no_protected = {"@context": {"name": "http://example.org/name"}}
assert "name" not in protected_terms(ctx_no_protected["@context"])

# Test @nest processing
nested_data = {
    "@context": {"container": "@nest"},
    "container": {
        "name": "Ada Lovelace",
        "age": 36
    }
}
processed = process_nest(nested_data["@context"], nested_data)
assert "name" in processed, "Name should be moved out of nested container"
assert "age" in processed, "Age should be moved out of nested container"
assert "container" not in processed, "Container should be removed"

# Test context propagation
ctx_propagate = {"@propagate": True}
ctx_no_propagate = {"@propagate": False}
proc = ContextProcessor()
assert proc.propagates(ctx_propagate) == True
assert proc.propagates(ctx_no_propagate) == False

# Test type-scoped contexts
entity_with_type_ctx = {
    "@context": {
        "@type": {
            "Person": {"name": "http://example.org/personName"},
            "Organization": {"name": "http://example.org/orgName"}
        }
    },
    "@type": "Person",
    "name": "Ada"
}
scoped_entity = proc.apply_scoped_contexts(entity_with_type_ctx)
assert isinstance(scoped_entity["@context"], list), "Should have a list context after applying scoped contexts"

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()